# Try GraphPFN on SlaveVoyages

[GraphPFN](https://github.com/yandex-research/graphpfn) is a pretrained model for data shaped as a graph. Here each voyage is a node and voyages are linked when they probably belong together (same ship within 10 years, same first owner, same first captain). It is scored on exactly the same hidden voyages as the other models.

**Runtime > Change runtime type > GPU.** GraphPFN needs Python 3.11 or newer, a CUDA GPU, and an older PyTorch (below 2.5). If Colab's Python is older than 3.11, GraphPFN will not install; the error will say so. You need the data file `tastdb-exp-2019.csv` (see DATA.md in the repo).

## 1. Get the code and install

In [ ]:
REPO_URL = "https://github.com/Ememobong28/slavevoyages-project.git"
!git clone -q {REPO_URL} repo
%cd repo
!python --version
!pip install -q -r requirements.txt
!pip install -q "torch==2.4" --index-url https://download.pytorch.org/whl/cu118
!pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.4/cu118/repo.html
!pip install -q graphpfn

## 2. Add the data and build the database

In [ ]:
from google.colab import files
import os
if not os.path.exists('tastdb-exp-2019.csv'):
    files.upload()                                   # choose tastdb-exp-2019.csv
!python voyage_db.py sqlite tastdb-exp-2019.csv voyages.sqlite
!python tests/test_framework.py

## 3. One quick run (tonnage, random split)

`graph='none'` is GraphPFN with no links. The other two add links. Comparing them shows what the links add.

In [ ]:
from graphpfn_run import run_graphpfn
import pandas as pd
rows = []
for graph in ['none', 'ship10', 'ship10_owner_captain']:
    rows.append(run_graphpfn('TONNAGE', 'random', seed=42, graph=graph, csv='voyages.sqlite', outdir='results/graphpfn'))
pd.concat(rows)[['model', 'median_APE_pct', 'within_25_pct', 'MAE_tons', 'settings', 'seconds']].round(3)

## 4. All five splits, tonnage and nationality, in-context

Uses the same five random splits (seeds 42-46) and the time-based split as the other models. Takes a while on a free GPU; start with one graph if short on time.

In [ ]:
import pandas as pd
out = []
for target in ['TONNAGE', 'NATIONAL_GROUP']:
    for scheme in ['random', 'temporal']:
        for seed in range(42, 47 if scheme == 'random' else 43):
            for graph in ['none', 'ship10', 'ship10_owner_captain']:
                out.append(run_graphpfn(target, scheme, seed=seed, graph=graph, csv='voyages.sqlite', outdir='results/graphpfn'))
allr = pd.concat(out, ignore_index=True); allr.to_csv('results/graphpfn/all_runs.csv', index=False)
cols = ['median_APE_pct', 'within_25_pct', 'MAE_tons', 'accuracy', 'macro_f1']
allr.groupby(['target', 'split', 'model'])[[c for c in cols if c in allr]].mean().round(3)

## 5. Optional: fine-tuning

Fine-tuning updates the model on this data. It is slower and is the setting the GraphPFN authors recommend.

In [ ]:
run_graphpfn('TONNAGE', 'random', seed=42, graph='ship10_owner_captain', finetune=True, csv='voyages.sqlite', outdir='results/graphpfn')

## 6. Download the results

Send the zip back or commit its contents to `results/graphpfn/`.

In [ ]:
!zip -qr graphpfn_results.zip results/graphpfn
files.download('graphpfn_results.zip')